In [ ]:
##################### Phase 4: Feature Engineering (Well-Level Pivot) #####################

import pandas as pd

# 1. Create a binary column: 1 if Positive ('+'), 0 if Negative
clean_df['Is_Positive'] = (clean_df['Final_Call'] == '+').astype(int)

# 2. Pivot the table so each Well gets its own row, and Targets become columns
well_level_df = clean_df.pivot_table(
    index=['Run_ID', 'Well', 'Source_File'], 
    columns='Target', 
    values='Is_Positive',
    aggfunc='max',
    fill_value=0
).reset_index()

# 3. Calculate Total HPV Infections per well (ignoring the CY5 Internal Control)
well_level_df['Total_HPV_Hits'] = well_level_df[['FAM', 'HEX', 'ROX']].sum(axis=1)

# 4.Remove Empty/Invalid Wells 
# We keep only rows where CY5 is 1 (Valid) OR Total_HPV_Hits is > 0 (True Positive)
patient_only_df = well_level_df[~((well_level_df['Total_HPV_Hits'] == 0) & (well_level_df['CY5'] == 0))].copy()

# 5. Preview the new clean structure and report the drops
print("--- Well-Level Clinical Data (True Patients Only) ---")
display(patient_only_df.head(10))

print("\n--- Data Cleaning Summary ---")
print(f"Original Dataset Size: {len(well_level_df)} total wells processed")
print(f"Clean Patient Dataset Size: {len(patient_only_df)} valid patient samples")
print(f"Removed {len(well_level_df) - len(patient_only_df)} empty/invalid wells (Likely NTCs).")

# 6. Show a quick epidemiological breakdown of Co-infections on the CLEAN data
print("\n--- Co-infection Summary (Valid Patients Only) ---")
print(patient_only_df['Total_HPV_Hits'].value_counts().sort_index())

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

# 1. Prepare the Data from the CLEANED dataset
infection_counts = patient_only_df['Total_HPV_Hits'].value_counts().sort_index()

# Data for the Donut Chart (Overall Positivity)
total_negative = infection_counts[0]
total_positive = infection_counts[1:].sum() # Sum of 1, 2, and 3 hits

# Data for the Bar Chart (Positive Breakdown Only)
positive_breakdown = infection_counts[1:].reset_index()
positive_breakdown.columns = ['Total_HPV_Hits', 'Count']
positive_breakdown['Category'] = ['Single Infection (1)', 'Co-infection (2)', 'Positive Controls (3)']

# 2. Set up the Figure with Subplots (1 row, 2 columns)
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# --- DONUT CHART (Left) ---
labels = ['Negative (0 Hits)', 'Positive (1+ Hits)']
sizes = [total_negative, total_positive]
colors = ['#d3d3d3', '#2c7fb8'] 

wedges, texts, autotexts = axes[0].pie(
    sizes, labels=labels, autopct='%1.1f%%', startangle=90, colors=colors, 
    textprops=dict(color="black", fontsize=12), wedgeprops=dict(width=0.4, edgecolor='w')
)
axes[0].set_title('True Patient HPV Positivity Rate', fontsize=14, fontweight='bold')

# --- BAR CHART (Right) ---
sns.barplot(
    data=positive_breakdown, x='Category', y='Count', 
    palette=['#41b6c4', '#2c7fb8', '#253494'], ax=axes[1]
)
axes[1].set_title('Breakdown of Positive Samples', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Infection Complexity', fontsize=12)
axes[1].set_ylabel('Number of Samples', fontsize=12)

for p in axes[1].patches:
    axes[1].annotate(format(p.get_height(), '.0f'), 
                     (p.get_x() + p.get_width() / 2., p.get_height()), 
                     ha = 'center', va = 'center', 
                     xytext = (0, 9), 
                     textcoords = 'offset points',
                     fontsize=11, fontweight='bold')

plt.tight_layout()
plt.show()